# Merchant / Recipient Name Normalization

Applies `merchant_normalizer.py` (bank-agnostic, two-tier: exact UPI_ID grouping + fuzzy name clustering) to `CSVS/4yrs_Clean_v2.xlsx` — the output of the improved multi-rule `Segregation.ipynb` extractor (better Bank/UPI_ID/Recipient_Name coverage than the original `4yrs_Clean.xlsx`).

Produces a `Recipient_Canonical` column and exports `CSVS/4yrs_Clean_v2_Merchants.xlsx`. Source files are left untouched.

Review the printed clusters below before trusting the output — adjust `THRESHOLD` if merchants are over- or under-merged.

In [ ]:
import os
import sys

# merchant_normalizer.py now lives in ml_service/app/services/ -- add ml_service/
# to sys.path so it's importable regardless of the Jupyter working directory.
ML_SERVICE_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', 'ml_service'))
if ML_SERVICE_ROOT not in sys.path:
    sys.path.insert(0, ML_SERVICE_ROOT)

import pandas as pd
from app.services.merchant_normalizer import SENTINELS, basic_normalize, normalize_recipients

In [4]:
df = pd.read_excel("CSVS\SpendWise_4yrs_Clean.xlsx")
print(df.shape)
df.head()

(1917, 12)


,Transaction_Date,Debit,Credit,Balance,Transaction_Mode,DR/CR_Indicator,Transaction_ID,Recipient_Name,Bank,UPI_ID,Note,Amount
0,2023-04-01,0.0,1500.0,1614.48,IMPS,CR,309111290781,Son,NaN,NaN,NaN,1500.0
1,2023-04-01,75.0,0.0,1539.48,UPI,DR,309122218462,ASIM HEM,HDFC,asimshah,UPI,-75.0
2,2023-04-02,1102.0,0.0,437.48,UPI,DR,309252494278,SAI RAJ,PYTM,paytmqr2,UPI,-1102.0
3,2023-04-02,10.0,0.0,427.48,UPI,DR,309255302589,PRACHI S,SBIN,prachi24,UPI,-10.0
4,2023-04-02,0.0,1500.0,1927.48,UPI,CR,309220310152,PRACHI S,SBIN,prachisw,NaN,1500.0


In [5]:
THRESHOLD = 90

df["Recipient_Canonical"] = normalize_recipients(df, threshold=THRESHOLD)
df[["Recipient_Name", "Recipient_Canonical"]].sample(15, random_state=1)

,Recipient_Name,Recipient_Canonical
48,Son,SON
1114,SAMEER B ALIRAM,SAMEER B ALIRAM
1210,JULFIKAR,JULFIKAR
194,FUNTER S,FUNTER S
368,Vishal O,VISHAL O
1221,ADITI SH,ADITI SH
1822,ROPPEN T,ROPPEN T
813,UNI AIMS,UNI AIMS
1176,We Care,WE CARE
555,Son,SON


## QA: inspect every cluster with more than one distinct name variant

Manually eyeball this list. Two failure modes to look for:
- **Over-merging**: clearly different merchants grouped under one canonical name → lower `THRESHOLD` won't help here, the UPI_ID/name data itself is ambiguous, or `THRESHOLD` is too low.
- **Under-merging**: obvious variants of the same merchant left in separate clusters → raise `THRESHOLD` down (stricter) or investigate why (e.g. very different spellings).

In [6]:
df["_basic"] = df["Recipient_Name"].apply(basic_normalize)

clusters = (
    df[~df["Recipient_Canonical"].isin(SENTINELS)]
    .groupby("Recipient_Canonical")["_basic"]
    .agg(lambda s: sorted(set(s)))
)
multi_member = clusters[clusters.apply(len) > 1]

print(f"{len(multi_member)} clusters with more than one distinct name variant:\n")
for canonical, variants in multi_member.items():
    print(f"{canonical!r:35}  <-  {variants}")

12 clusters with more than one distinct name variant:

'AASHAY M'                           <-  ['AASHAY M', 'AASHAYJ2']
'ABUTURAB'                           <-  ['ABU TURA', 'ABUTURAB']
'ALPHA VI'                           <-  ['ALPHA VI', 'ALPHARAN']
'GOOGLE I'                           <-  ['GOOGLE I', 'GOOGLEPAY']
'MAMTA RA'                           <-  ['MAMTA RA', 'MAMTAPAT']
'MAYANK R'                           <-  ['MAYANK R', 'MAYANKRA']
'MR VIHAA'                           <-  ['MR VIHAA', 'VIHAANSH']
'PRADHAN'                            <-  ['PRADHAN', 'PRASHANT']
'PRATHAM'                            <-  ['PRATHAM', 'PRATHAMS']
'SHLOK SA'                           <-  ['SHLOK SA', 'SHLOKSM2']
'SHUBHAM'                            <-  ['SHUBHAM', 'SHUBHAMR']
'TRA NSAFER'                         <-  ['TRA NSAFER', 'TRA NSFER']


## Manual review: truncation-prefix candidates and known aliases

`find_prefix_variants` catches truncation artifacts the two automated tiers above miss — cases where one canonical name is a literal prefix of another (e.g. `"AIRTEL"` vs `"AIRTEL P"`). It's not auto-merged: some hits are genuinely ambiguous without knowing your actual contacts (e.g. common first names like "Krishna"), and a blind fuzzy-similarity merge at this string length was tested and rejected — see `merchant_normalizer.find_prefix_variants` docstring for why.

Review the printed candidates below, then `manual_aliases` applies the pairs already confirmed as the same real payee.

In [ ]:
from app.services.merchant_normalizer import find_prefix_variants

names = [n for n in df["Recipient_Canonical"].unique() if n not in SENTINELS]
prefix_pairs = find_prefix_variants(names)

print(f"{len(prefix_pairs)} truncation-prefix candidates (not auto-merged — review each):\n")
for short, long_ in prefix_pairs:
    print(f"{short!r:20} <- prefix of -> {long_!r}")

In [ ]:
manual_aliases = {
    # brands
    "AIRTEL P": "AIRTEL",
    "GOOGLE P": "GOOGLE PAY",
    "ZOMATO L": "ZOMATO",
    "ZOMATO O": "ZOMATO",
    # personal / organization names
    "ATHARVAH": "ATHARVA",
    "SAMEER B": "SAMEER B ALIRAM",
    "TANISH NX": "TANISH N",
    "SRI LAKS": "SRI LAKSHMI NARSIMHA PTHANE",
    # confirmed on 2026-07-11 — KRISHNAP intentionally excluded (not confirmed as same person)
    "KRISHNAM": "KRISHNA",
    "PRACHI S": "MRS. PRACHI SAMEER SAW",
}
df["Recipient_Canonical"] = df["Recipient_Canonical"].replace(manual_aliases)
print("Unique Recipient_Canonical after manual aliases:", df["Recipient_Canonical"].nunique())

In [7]:
assert len(df) == 1917, f"Row count changed unexpectedly: {len(df)}"
for sentinel in SENTINELS:
    mask = df["Recipient_Name"] == sentinel
    if mask.any():
        assert (df.loc[mask, "Recipient_Canonical"] == sentinel).all(), f"{sentinel} got altered"

print("Sanity checks passed.")
print("Unique Recipient_Name:", df["Recipient_Name"].nunique())
print("Unique Recipient_Canonical:", df["Recipient_Canonical"].nunique())

Sanity checks passed.
Unique Recipient_Name: 505
Unique Recipient_Canonical: 467


In [8]:
out = df.drop(columns=["_basic"])
out.to_excel("CSVS/SpendWise_4yrs_Clean_Merchants.xlsx", index=False)
print("Saved CSVS/SpendWise_4yrs_Clean_Merchants.xlsx")

Saved CSVS/SpendWise_4yrs_Clean_Merchants.xlsx
